In [5]:
"""
Order Book Simulation with Sanity Checks and Dashboarding
Day 5 Deliverable: Comprehensive Simulation Pipeline
"""

!pip install mplfinance
!pip install yfinance
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import mplfinance as mpf
import random
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass, field
from collections import defaultdict, deque
import logging

# ============================================================================
# 1. DATA STRUCTURES
# ============================================================================

@dataclass(order=True)
class Order:
    """Order data structure with validation"""
    order_id: int
    timestamp: datetime
    is_buy: bool
    price: float
    quantity: int
    order_type: str = 'LIMIT'  # LIMIT or MARKET

    def __post_init__(self):
        """Validate order parameters on creation"""
        assert self.price >= 0, f"Negative price: {self.price}"
        assert self.quantity > 0, f"Non-positive quantity: {self.quantity}"
        assert self.order_type in ['LIMIT', 'MARKET'], f"Invalid order type: {self.order_type}"

@dataclass
class Trade:
    """Trade data structure"""
    trade_id: int
    timestamp: datetime
    price: float
    quantity: int
    buyer_id: Optional[int] = None
    seller_id: Optional[int] = None

    def __post_init__(self):
        """Validate trade parameters"""
        assert self.price >= 0, f"Negative trade price: {self.price}"
        assert self.quantity > 0, f"Non-positive trade quantity: {self.quantity}"

@dataclass
class OrderBook:
    """Order book maintaining bids and asks"""
    bids: Dict[float, List[Order]] = field(default_factory=lambda: defaultdict(list))
    asks: Dict[float, List[Order]] = field(default_factory=lambda: defaultdict(list))
    tape: List[Trade] = field(default_factory=list)

    def add_order(self, order: Order) -> List[Trade]:
        """Add order to book, return list of resulting trades"""
        trades = []

        if order.is_buy:
            trades = self._match_buy_order(order)
            if order.quantity > 0:
                self._add_bid(order)
        else:
            trades = self._match_sell_order(order)
            if order.quantity > 0:
                self._add_ask(order)

        return trades

    def _match_buy_order(self, order: Order) -> List[Trade]:
        """Match buy order against existing asks"""
        trades = []

        # Get sorted ask prices (lowest first for buy orders)
        ask_prices = sorted(self.asks.keys())

        for ask_price in ask_prices:
            if order.quantity == 0 or (order.price < ask_price and order.order_type == 'LIMIT'):
                break

            while self.asks[ask_price] and order.quantity > 0:
                ask_order = self.asks[ask_price][0]

                # Determine trade price and quantity
                trade_price = ask_price  # Price is the ask price
                trade_quantity = min(order.quantity, ask_order.quantity)

                # Create trade
                trade = Trade(
                    trade_id=len(self.tape) + len(trades) + 1,
                    timestamp=order.timestamp,
                    price=trade_price,
                    quantity=trade_quantity,
                    buyer_id=order.order_id,
                    seller_id=ask_order.order_id
                )
                trades.append(trade)

                # Update quantities
                order.quantity -= trade_quantity
                ask_order.quantity -= trade_quantity

                # Remove ask order if fully filled
                if ask_order.quantity == 0:
                    self.asks[ask_price].pop(0)

            # Remove price level if empty
            if not self.asks[ask_price]:
                del self.asks[ask_price]

        return trades

    def _match_sell_order(self, order: Order) -> List[Trade]:
        """Match sell order against existing bids"""
        trades = []

        # Get sorted bid prices (highest first for sell orders)
        bid_prices = sorted(self.bids.keys(), reverse=True)

        for bid_price in bid_prices:
            if order.quantity == 0 or (order.price > bid_price and order.order_type == 'LIMIT'):
                break

            while self.bids[bid_price] and order.quantity > 0:
                bid_order = self.bids[bid_price][0]

                # Determine trade price and quantity
                trade_price = bid_price  # Price is the bid price
                trade_quantity = min(order.quantity, bid_order.quantity)

                # Create trade
                trade = Trade(
                    trade_id=len(self.tape) + len(trades) + 1,
                    timestamp=order.timestamp,
                    price=trade_price,
                    quantity=trade_quantity,
                    buyer_id=bid_order.order_id,
                    seller_id=order.order_id
                )
                trades.append(trade)

                # Update quantities
                order.quantity -= trade_quantity
                bid_order.quantity -= trade_quantity

                # Remove bid order if fully filled
                if bid_order.quantity == 0:
                    self.bids[bid_price].pop(0)

            # Remove price level if empty
            if not self.bids[bid_price]:
                del self.bids[bid_price]

        return trades

    def _add_bid(self, order: Order):
        """Add bid to order book"""
        self.bids[order.price].append(order)

    def _add_ask(self, order: Order):
        """Add ask to order book"""
        self.asks[order.price].append(order)

    def get_best_bid(self) -> Optional[float]:
        """Get best (highest) bid price"""
        return max(self.bids.keys()) if self.bids else None

    def get_best_ask(self) -> Optional[float]:
        """Get best (lowest) ask price"""
        return min(self.asks.keys()) if self.asks else None

    def get_spread(self) -> Optional[float]:
        """Get bid-ask spread"""
        best_bid = self.get_best_bid()
        best_ask = self.get_best_ask()

        if best_bid is not None and best_ask is not None:
            return best_ask - best_bid
        return None

    def get_mid_price(self) -> Optional[float]:
        """Get mid price"""
        best_bid = self.get_best_bid()
        best_ask = self.get_best_ask()

        if best_bid is not None and best_ask is not None:
            return (best_bid + best_ask) / 2
        return None

    def get_depth(self, price: float) -> int:
        """Get total quantity at price level"""
        bid_qty = sum(order.quantity for order in self.bids.get(price, []))
        ask_qty = sum(order.quantity for order in self.asks.get(price, []))
        return bid_qty + ask_qty

# ============================================================================
# 2. SANITY CHECKER (VALIDATION LAYER)
# ============================================================================

class SanityChecker:
    """Aggressive assertion-based validation layer"""

    @staticmethod
    def validate_order_book(book: OrderBook, timestamp: datetime,
                           trades_just_executed: List[Trade] = None):
        """
        Validate all core invariants of the order book

        Reference: https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement
        """
        trades_just_executed = trades_just_executed or []

        # 1. Validate prices are non-negative
        for price in book.bids.keys():
            assert price >= 0, f"Negative bid price: {price} at {timestamp}"
        for price in book.asks.keys():
            assert price >= 0, f"Negative ask price: {price} at {timestamp}"

        # 2. Validate quantities are non-negative
        for price, orders in book.bids.items():
            for order in orders:
                assert order.quantity >= 0, f"Negative bid quantity: {order.quantity} at {price}"
        for price, orders in book.asks.items():
            for order in orders:
                assert order.quantity >= 0, f"Negative ask quantity: {order.quantity} at {price}"

        # 3. Validate order book depth is non-negative
        for price in set(list(book.bids.keys()) + list(book.asks.keys())):
            depth = book.get_depth(price)
            assert depth >= 0, f"Negative depth at price {price}: {depth}"

        # 4. Validate bid < ask except during trade execution
        best_bid = book.get_best_bid()
        best_ask = book.get_best_ask()

        if best_bid is not None and best_ask is not None:
            if not trades_just_executed:
                # No trades just executed, must have bid < ask
                assert best_bid < best_ask, (
                    f"Bid {best_bid} >= Ask {best_ask} without trade at {timestamp}"
                )
            else:
                # Trades just executed, bid may cross ask temporarily
                # But should recover quickly in next validation
                pass

        # 5. Validate tape is append-only (check sequence numbers)
        for i in range(1, len(book.tape)):
            assert book.tape[i].trade_id > book.tape[i-1].trade_id, (
                f"Tape not append-only at position {i}"
            )

        # 6. Validate trades just executed
        for trade in trades_just_executed:
            assert trade.price >= 0, f"Negative trade price in execution: {trade.price}"
            assert trade.quantity > 0, f"Non-positive trade quantity: {trade.quantity}"

    @staticmethod
    def validate_trade_tape(tape: List[Trade]):
        """Validate entire trade tape"""
        for trade in tape:
            assert trade.price >= 0, f"Negative price in tape: {trade.price}"
            assert trade.quantity > 0, f"Non-positive quantity in tape: {trade.quantity}"
            assert trade.timestamp is not None, "Trade missing timestamp"

        # Check chronological order
        for i in range(1, len(tape)):
            assert tape[i].timestamp >= tape[i-1].timestamp, (
                f"Tape not chronological: {tape[i].timestamp} < {tape[i-1].timestamp}"
            )

# ============================================================================
# 3. ORDER GENERATOR (TEST HARNESS)
# ============================================================================

class OrderGenerator:
    """
    Deterministic random order generator

    Reference: https://docs.python.org/3/library/random.html
    """

    def __init__(self, seed: int = 42):
        """Initialize with fixed seed for reproducibility"""
        random.seed(seed)
        np.random.seed(seed)

        # Base price for the simulation
        self.base_price = 100.0

        # Order ID counter
        self.order_id = 0

        # Statistical distributions for order generation
        # Based on market microstructure research:
        # - Price offsets: Normal distribution (centered around fair value)
        # - Order sizes: Log-normal distribution (typical for financial markets)
        # - Inter-arrival times: Exponential distribution (common for Poisson processes)

        self.price_std = 5.0  # Standard deviation for price offsets
        self.size_mean = 100  # Mean order size
        self.size_std = 50   # Std dev for order sizes
        self.arrival_rate = 10  # Orders per minute

    def generate_order(self, timestamp: datetime) -> Order:
        """Generate a random order"""
        self.order_id += 1

        # Determine order type (90% LIMIT, 10% MARKET)
        order_type = 'LIMIT' if random.random() < 0.9 else 'MARKET'

        # Determine buy/sell (50/50 split)
        is_buy = random.random() < 0.5

        # Generate price based on distribution
        if order_type == 'LIMIT':
            # LIMIT orders: Price follows normal distribution around base price
            # With slight bias toward best bid/ask
            price_offset = np.random.normal(0, self.price_std)
            price = max(0.01, round(self.base_price + price_offset, 2))
        else:
            # MARKET orders: Price is None (will execute at best available)
            price = 0.0  # Will be ignored for market orders

        # Generate quantity using log-normal distribution
        # Log-normal is common for trade sizes (positive, right-skewed)
        quantity = int(max(1, np.random.lognormal(
            mean=np.log(self.size_mean),
            sigma=np.log(1 + self.size_std/self.size_mean)
        )))

        # Add some clustering (round lots of 10)
        quantity = (quantity // 10) * 10

        return Order(
            order_id=self.order_id,
            timestamp=timestamp,
            is_buy=is_buy,
            price=price,
            quantity=quantity,
            order_type=order_type
        )

    def generate_order_sequence(self, n_orders: int = 1000,
                               start_time: datetime = None) -> List[Order]:
        """Generate a sequence of random orders"""
        if start_time is None:
            start_time = datetime.now()

        orders = []
        current_time = start_time

        for _ in range(n_orders):
            # Generate inter-arrival time (exponential distribution)
            # Common for Poisson arrival processes in markets
            inter_arrival = np.random.exponential(60 / self.arrival_rate)
            current_time += timedelta(seconds=inter_arrival)

            order = self.generate_order(current_time)
            orders.append(order)

        return orders

# ============================================================================
# 4. SIMULATION ENGINE
# ============================================================================

class SimulationEngine:
    """Main simulation engine"""

    def __init__(self, initial_price: float = 100.0, seed: int = 42):
        self.order_book = OrderBook()
        self.order_generator = OrderGenerator(seed)
        self.sanity_checker = SanityChecker()
        self.l1_snapshots = []  # For storing best bid/ask over time
        self.all_orders = []    # For tracking all orders

        # Statistics
        self.stats = {
            'total_orders': 0,
            'total_trades': 0,
            'total_volume': 0,
            'max_spread': 0,
            'min_spread': float('inf'),
            'validation_errors': 0
        }

        # Initialize with some orders
        self._initialize_book(initial_price)

    def _initialize_book(self, initial_price: float):
        """Initialize order book with some resting orders"""
        timestamp = datetime.now()

        # Add initial bids
        for i in range(5):
            bid_price = initial_price - (i + 1) * 0.5
            order = Order(
                order_id=self.order_generator.order_id + 1,
                timestamp=timestamp,
                is_buy=True,
                price=bid_price,
                quantity=random.randint(100, 500),
                order_type='LIMIT'
            )
            self.order_book.add_order(order)
            self.order_generator.order_id += 1

        # Add initial asks
        for i in range(5):
            ask_price = initial_price + (i + 1) * 0.5
            order = Order(
                order_id=self.order_generator.order_id + 1,
                timestamp=timestamp,
                is_buy=False,
                price=ask_price,
                quantity=random.randint(100, 500),
                order_type='LIMIT'
            )
            self.order_book.add_order(order)
            self.order_generator.order_id += 1

        # Record initial L1 snapshot
        self._record_l1_snapshot(timestamp)

    def _record_l1_snapshot(self, timestamp: datetime):
        """Record current best bid/ask"""
        best_bid = self.order_book.get_best_bid()
        best_ask = self.order_book.get_best_ask()
        spread = self.order_book.get_spread()
        mid_price = self.order_book.get_mid_price()

        self.l1_snapshots.append({
            'timestamp': timestamp,
            'best_bid': best_bid,
            'best_ask': best_ask,
            'spread': spread,
            'mid_price': mid_price
        })

        # Update spread statistics
        if spread is not None:
            self.stats['max_spread'] = max(self.stats['max_spread'], spread)
            self.stats['min_spread'] = min(self.stats['min_spread'], spread)

    def run(self, n_orders: int = 1000) -> pd.DataFrame:
        """Run simulation with n_orders"""
        print(f"Starting simulation with {n_orders} orders...")

        # Generate orders
        orders = self.order_generator.generate_order_sequence(n_orders)
        self.all_orders = orders

        # Process orders
        for i, order in enumerate(orders):
            try:
                # Process order
                trades = self.order_book.add_order(order)

                # Update statistics
                self.stats['total_orders'] += 1
                self.stats['total_trades'] += len(trades)
                self.stats['total_volume'] += sum(t.quantity for t in trades)

                # Record L1 snapshot
                self._record_l1_snapshot(order.timestamp)

                # Aggressive sanity checking
                self.sanity_checker.validate_order_book(
                    self.order_book,
                    order.timestamp,
                    trades_just_executed=trades
                )

                # Add trades to tape
                self.order_book.tape.extend(trades)

                # Progress reporting
                if (i + 1) % 100 == 0:
                    print(f"  Processed {i + 1}/{n_orders} orders...")

            except AssertionError as e:
                self.stats['validation_errors'] += 1
                print(f"  Validation error at order {i + 1}: {e}")
                # Continue despite error for demonstration

        # Final validation
        self.sanity_checker.validate_trade_tape(self.order_book.tape)

        print(f"Simulation complete!")
        print(f"  Total orders: {self.stats['total_orders']}")
        print(f"  Total trades: {self.stats['total_trades']}")
        print(f"  Total volume: {self.stats['total_volume']}")
        print(f"  Validation errors: {self.stats['validation_errors']}")

        return self._prepare_dataframes()

    def _prepare_dataframes(self) -> Dict[str, pd.DataFrame]:
        """Prepare DataFrames for analysis"""
        # Trade tape DataFrame
        if self.order_book.tape:
            tape_df = pd.DataFrame([
                {
                    'timestamp': t.timestamp,
                    'price': t.price,
                    'quantity': t.quantity,
                    'trade_id': t.trade_id
                }
                for t in self.order_book.tape
            ])
            tape_df.set_index('timestamp', inplace=True)
        else:
            tape_df = pd.DataFrame(columns=['price', 'quantity', 'trade_id'])

        # L1 snapshots DataFrame
        l1_df = pd.DataFrame(self.l1_snapshots)
        if not l1_df.empty:
            l1_df.set_index('timestamp', inplace=True)

        # Orders DataFrame
        orders_df = pd.DataFrame([
            {
                'timestamp': o.timestamp,
                'is_buy': o.is_buy,
                'price': o.price,
                'quantity': o.quantity,
                'order_type': o.order_type,
                'order_id': o.order_id
            }
            for o in self.all_orders
        ])
        if not orders_df.empty:
            orders_df.set_index('timestamp', inplace=True)

        return {
            'trades': tape_df,
            'l1_snapshots': l1_df,
            'orders': orders_df
        }

# ============================================================================
# 5. DASHBOARD GENERATOR (VISUALIZATION)
# ============================================================================

class DashboardGenerator:
    """
    Generate comprehensive dashboard visualizations

    References:
    - https://matplotlib.org/stable/users/index.html
    - https://github.com/matplotlib/mplfinance
    - https://matplotlib.org/stable/api/backend_pdf_api.html
    - https://in.tradingview.com/ (for design reference)
    """

    def __init__(self, style: str = 'seaborn-v0_8-darkgrid'):
        plt.style.use(style)
        self.fig_size = (16, 10)

    def create_candlestick_chart(self, ohlc_df: pd.DataFrame,
                                title: str = "Candlestick Chart") -> plt.Figure:
        """
        Create candlestick chart using mplfinance

        Reference: https://github.com/matplotlib/mplfinance
        """
        # Validate OHLC data
        assert not ohlc_df.empty, "OHLC DataFrame is empty"
        assert {'open', 'high', 'low', 'close'}.issubset(ohlc_df.columns), \
            "OHLC DataFrame missing required columns"

        # Create figure with custom style
        mc = mpf.make_marketcolors(
            up='green', down='red',
            edge={'up': 'green', 'down': 'red'},
            wick={'up': 'green', 'down': 'red'},
            volume='in'
        )

        s = mpf.make_mpf_style(
            marketcolors=mc,
            gridstyle='--',
            gridcolor='gray',
            facecolor='white'
        )

        fig, axes = mpf.plot(
            ohlc_df,
            type='candle',
            style=s,
            title=title,
            ylabel='Price',
            volume=True if 'volume' in ohlc_df.columns else False,
            figsize=self.fig_size,
            returnfig=True,
            show_nontrading=False
        )

        return fig

    def create_price_evolution_plot(self, l1_df: pd.DataFrame,
                                   trades_df: pd.DataFrame) -> plt.Figure:
        """Create price evolution plot with bid/ask and trades"""
        fig, ax = plt.subplots(figsize=self.fig_size)

        # Plot bid and ask prices
        if not l1_df.empty and 'best_bid' in l1_df.columns and 'best_ask' in l1_df.columns:
            ax.plot(l1_df.index, l1_df['best_bid'],
                   label='Best Bid', color='blue', alpha=0.7, linewidth=1)
            ax.plot(l1_df.index, l1_df['best_ask'],
                   label='Best Ask', color='red', alpha=0.7, linewidth=1)

            # Fill between bid and ask (spread)
            ax.fill_between(l1_df.index,
                           l1_df['best_bid'],
                           l1_df['best_ask'],
                           alpha=0.2, color='gray', label='Spread')

        # Plot trades
        if not trades_df.empty and 'price' in trades_df.columns:
            buy_trades = [t for t in trades_df.itertuples()
                         if hasattr(t, 'is_buy') and t.is_buy] if hasattr(trades_df.iloc[0], 'is_buy') else []
            sell_trades = [t for t in trades_df.itertuples()
                          if hasattr(t, 'is_buy') and not t.is_buy] if hasattr(trades_df.iloc[0], 'is_buy') else []

            if buy_trades:
                buy_times = [t.Index for t in buy_trades]
                buy_prices = [t.price for t in buy_trades]
                ax.scatter(buy_times, buy_prices, color='green',
                          marker='^', s=50, alpha=0.7, label='Buy Trades', zorder=5)

            if sell_trades:
                sell_times = [t.Index for t in sell_trades]
                sell_prices = [t.price for t in sell_trades]
                ax.scatter(sell_times, sell_prices, color='red',
                          marker='v', s=50, alpha=0.7, label='Sell Trades', zorder=5)

        ax.set_xlabel('Time')
        ax.set_ylabel('Price')
        ax.set_title('Price Evolution with Bid/Ask Spread')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Format x-axis for time
        fig.autofmt_xdate()

        return fig

    def create_spread_analysis_plot(self, l1_df: pd.DataFrame) -> plt.Figure:
        """Create spread analysis plot"""
        fig, axes = plt.subplots(2, 2, figsize=self.fig_size)

        # 1. Spread over time
        if not l1_df.empty and 'spread' in l1_df.columns:
            axes[0, 0].plot(l1_df.index, l1_df['spread'], color='purple', linewidth=1)
            axes[0, 0].set_title('Bid-Ask Spread Over Time')
            axes[0, 0].set_xlabel('Time')
            axes[0, 0].set_ylabel('Spread')
            axes[0, 0].grid(True, alpha=0.3)

            # Add horizontal line for average spread
            avg_spread = l1_df['spread'].mean()
            axes[0, 0].axhline(y=avg_spread, color='red', linestyle='--',
                              alpha=0.7, label=f'Avg: {avg_spread:.4f}')
            axes[0, 0].legend()

        # 2. Spread histogram
        if not l1_df.empty and 'spread' in l1_df.columns:
            axes[0, 1].hist(l1_df['spread'].dropna(), bins=30, color='skyblue',
                           edgecolor='black', alpha=0.7)
            axes[0, 1].set_title('Spread Distribution')
            axes[0, 1].set_xlabel('Spread')
            axes[0, 1].set_ylabel('Frequency')
            axes[0, 1].grid(True, alpha=0.3)

        # 3. Mid-price volatility (rolling)
        if not l1_df.empty and 'mid_price' in l1_df.columns:
            rolling_std = l1_df['mid_price'].rolling(window=10).std()
            axes[1, 0].plot(l1_df.index, rolling_std, color='orange', linewidth=1)
            axes[1, 0].set_title('Rolling Mid-Price Volatility (10-period)')
            axes[1, 0].set_xlabel('Time')
            axes[1, 0].set_ylabel('Volatility (std dev)')
            axes[1, 0].grid(True, alpha=0.3)

        # 4. Order book imbalance (simplified)
        if not l1_df.empty and 'best_bid' in l1_df.columns and 'best_ask' in l1_df.columns:
            # Calculate simple imbalance measure
            price_range = np.linspace(l1_df['best_bid'].min() - 1,
                                     l1_df['best_ask'].max() + 1, 20)
            # This is a placeholder - in real implementation, use actual depth
            axes[1, 1].bar(price_range, np.random.randn(len(price_range)) + 10,
                          alpha=0.7, color='teal')
            axes[1, 1].set_title('Order Book Depth Profile')
            axes[1, 1].set_xlabel('Price')
            axes[1, 1].set_ylabel('Depth')
            axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        return fig

    def create_order_flow_plot(self, orders_df: pd.DataFrame,
                              trades_df: pd.DataFrame) -> plt.Figure:
        """Create order flow analysis plot"""
        fig, axes = plt.subplots(2, 2, figsize=self.fig_size)

        if not orders_df.empty:
            # 1. Order size distribution
            axes[0, 0].hist(orders_df['quantity'], bins=30, color='lightcoral',
                           edgecolor='black', alpha=0.7)
            axes[0, 0].set_title('Order Size Distribution')
            axes[0, 0].set_xlabel('Order Size')
            axes[0, 0].set_ylabel('Frequency')
            axes[0, 0].grid(True, alpha=0.3)

            # 2. Buy vs Sell order ratio over time
            orders_df['hour'] = orders_df.index.hour
            if 'is_buy' in orders_df.columns:
                buy_ratio = orders_df.groupby('hour')['is_buy'].mean()
                axes[0, 1].bar(buy_ratio.index, buy_ratio.values,
                              color='lightgreen', alpha=0.7)
                axes[0, 1].set_title('Buy Order Ratio by Hour')
                axes[0, 1].set_xlabel('Hour of Day')
                axes[0, 1].set_ylabel('Buy Ratio')
                axes[0, 1].grid(True, alpha=0.3)

        if not trades_df.empty:
            # 3. Trade size vs price
            if 'quantity' in trades_df.columns and 'price' in trades_df.columns:
                scatter = axes[1, 0].scatter(trades_df.index, trades_df['price'],
                                            c=trades_df['quantity'],
                                            cmap='viridis', alpha=0.6, s=30)
                axes[1, 0].set_title('Trade Size vs Price')
                axes[1, 0].set_xlabel('Time')
                axes[1, 0].set_ylabel('Price')
                axes[1, 0].grid(True, alpha=0.3)
                plt.colorbar(scatter, ax=axes[1, 0], label='Trade Size')

            # 4. Cumulative volume
            if 'quantity' in trades_df.columns:
                cumulative_volume = trades_df['quantity'].cumsum()
                axes[1, 1].plot(trades_df.index, cumulative_volume,
                               color='darkblue', linewidth=2)
                axes[1, 1].set_title('Cumulative Trading Volume')
                axes[1, 1].set_xlabel('Time')
                axes[1, 1].set_ylabel('Cumulative Volume')
                axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        return fig

    def create_validation_report(self, stats: Dict,
                                ohlc_df: pd.DataFrame) -> plt.Figure:
        """Create validation statistics report"""
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # 1. Key statistics table
        ax = axes[0]
        ax.axis('tight')
        ax.axis('off')

        stats_table_data = [
            ['Total Orders', f"{stats.get('total_orders', 0):,}"],
            ['Total Trades', f"{stats.get('total_trades', 0):,}"],
            ['Total Volume', f"{stats.get('total_volume', 0):,}"],
            ['Max Spread', f"{stats.get('max_spread', 0):.4f}"],
            ['Min Spread', f"{stats.get('min_spread', 0):.4f}"],
            ['Validation Errors', f"{stats.get('validation_errors', 0)}"],
        ]

        table = ax.table(cellText=stats_table_data,
                        colLabels=['Metric', 'Value'],
                        cellLoc='left',
                        loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.5)
        ax.set_title('Simulation Statistics', fontsize=14, pad=20)

        # 2. OHLC validation checks
        ax = axes[1]
        validation_checks = []

        if not ohlc_df.empty:
            # Check 1: High >= Low
            high_low_check = (ohlc_df['high'] >= ohlc_df['low']).all()
            validation_checks.append(['High >= Low', '✓' if high_low_check else '✗'])

            # Check 2: High >= Open and High >= Close
            high_open_check = (ohlc_df['high'] >= ohlc_df['open']).all()
            high_close_check = (ohlc_df['high'] >= ohlc_df['close']).all()
            validation_checks.append(['High >= Open/Close', '✓' if high_open_check and high_close_check else '✗'])

            # Check 3: Low <= Open and Low <= Close
            low_open_check = (ohlc_df['low'] <= ohlc_df['open']).all()
            low_close_check = (ohlc_df['low'] <= ohlc_df['close']).all()
            validation_checks.append(['Low <= Open/Close', '✓' if low_open_check and low_close_check else '✗'])

            # Check 4: No zero-volume candles (if available)
            if 'volume' in ohlc_df.columns:
                zero_volume = (ohlc_df['volume'] == 0).sum()
                validation_checks.append(['Zero-volume candles', f'{zero_volume}'])

            # Check 5: Price continuity
            if len(ohlc_df) > 1:
                price_range = ohlc_df['high'].max() - ohlc_df['low'].min()
                validation_checks.append(['Price Range', f'{price_range:.2f}'])

        validation_checks.append(['All Sanity Checks', 'PASSED' if stats.get('validation_errors', 0) == 0 else 'FAILED'])

        ax.axis('tight')
        ax.axis('off')
        table = ax.table(cellText=validation_checks,
                        colLabels=['Check', 'Result'],
                        cellLoc='left',
                        loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.5)
        ax.set_title('OHLC Validation Checks', fontsize=14, pad=20)

        plt.tight_layout()
        return fig

    def generate_pdf_report(self, dataframes: Dict[str, pd.DataFrame],
                           stats: Dict, output_path: str = 'simulation_report.pdf'):
        """
        Generate comprehensive PDF report

        Reference: https://matplotlib.org/stable/api/backend_pdf_api.html
        """
        print(f"Generating PDF report: {output_path}")

        with PdfPages(output_path) as pdf:
            # Page 1: Title and summary
            fig, ax = plt.subplots(figsize=(16, 10))
            ax.axis('off')

            title_text = "Order Book Simulation Report\n\n"
            summary_text = (
                f"Simulation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
                f"Total Orders Processed: {stats.get('total_orders', 0):,}\n"
                f"Total Trades Executed: {stats.get('total_trades', 0):,}\n"
                f"Total Volume: {stats.get('total_volume', 0):,}\n"
                f"Validation Status: {'PASSED' if stats.get('validation_errors', 0) == 0 else 'FAILED'}\n\n"
                "This report contains comprehensive analysis of the order book simulation,\n"
                "including candlestick charts, spread analysis, and order flow metrics."
            )

            ax.text(0.5, 0.7, title_text, fontsize=24, fontweight='bold',
                   ha='center', va='center', transform=ax.transAxes)
            ax.text(0.5, 0.4, summary_text, fontsize=14,
                   ha='center', va='center', transform=ax.transAxes,
                   linespacing=1.5)

            pdf.savefig(fig, bbox_inches='tight')
            plt.close()

            # Page 2: Candlestick chart (if we have OHLC data)
            if 'trades' in dataframes and not dataframes['trades'].empty:
                # Resample to 1-minute OHLC using Pandas
                # Reference: https://pandas.pydata.org/docs/user_guide/timeseries.html#resampling
                trades_df = dataframes['trades']

                if len(trades_df) > 0:
                    # Resample to 1-minute intervals
                    ohlc_df = trades_df['price'].resample('1min').ohlc()
                    volume_df = trades_df['quantity'].resample('1min').sum()

                    # Combine OHLC with volume
                    ohlc_df['volume'] = volume_df

                    # Fill NaN values (forward fill for OHLC, 0 for volume)
                    ohlc_df[['open', 'high', 'low', 'close']] = \
                        ohlc_df[['open', 'high', 'low', 'close']].ffill()
                    ohlc_df['volume'] = ohlc_df['volume'].fillna(0)

                    # Create candlestick chart
                    fig = self.create_candlestick_chart(
                        ohlc_df,
                        title="1-Minute Candlestick Chart (from Trade Tape)"
                    )
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close()

            # Page 3: Price evolution and spread
            if 'l1_snapshots' in dataframes and 'trades' in dataframes:
                fig = self.create_price_evolution_plot(
                    dataframes['l1_snapshots'],
                    dataframes['trades']
                )
                pdf.savefig(fig, bbox_inches='tight')
                plt.close()

            # Page 4: Spread analysis
            if 'l1_snapshots' in dataframes:
                fig = self.create_spread_analysis_plot(dataframes['l1_snapshots'])
                pdf.savefig(fig, bbox_inches='tight')
                plt.close()

            # Page 5: Order flow analysis
            if 'orders' in dataframes and 'trades' in dataframes:
                fig = self.create_order_flow_plot(
                    dataframes['orders'],
                    dataframes['trades']
                )
                pdf.savefig(fig, bbox_inches='tight')
                plt.close()

            # Page 6: Validation report
            if 'trades' in dataframes and not dataframes['trades'].empty:
                # Create OHLC for validation
                trades_df = dataframes['trades']
                if len(trades_df) > 0:
                    ohlc_df = trades_df['price'].resample('1min').ohlc()
                    fig = self.create_validation_report(stats, ohlc_df)
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close()

            # Add metadata to PDF
            pdf.infodict()['Title'] = 'Order Book Simulation Report'
            pdf.infodict()['Author'] = 'Trading Simulation System'
            pdf.infodict()['Subject'] = 'Day 5 Deliverable: Simulation Dashboard'
            pdf.infodict()['Keywords'] = 'Trading, Order Book, Simulation, Candlestick'
            pdf.infodict()['CreationDate'] = datetime.now()
            pdf.infodict()['ModDate'] = datetime.now()

        print(f"PDF report generated successfully: {output_path}")

# ============================================================================
# 6. MAIN SIMULATION SCRIPT
# ============================================================================

def run_simulation():
    """Main simulation function"""
    print("=" * 70)
    print("ORDER BOOK SIMULATION - DAY 5 DELIVERABLE")
    print("=" * 70)

    # 1. Seed RNG for reproducibility
    SEED = 42
    N_ORDERS = 1000
    OUTPUT_PDF = 'simulation_report.pdf'

    print(f"Configuration:")
    print(f"  Seed: {SEED}")
    print(f"  Orders to generate: {N_ORDERS}")
    print(f"  Output PDF: {OUTPUT_PDF}")
    print()

    # 2. Initialize simulation engine
    engine = SimulationEngine(initial_price=100.0, seed=SEED)

    # 3. Generate and process random orders
    dataframes = engine.run(n_orders=N_ORDERS)

    # 4. Perform OHLC validation
    print("\nPerforming OHLC validation checks...")

    if not dataframes['trades'].empty:
        # Resample to 1-minute OHLC using Pandas
        # Reference: https://pandas.pydata.org/docs/user_guide/timeseries.html#resampling
        trades_df = dataframes['trades']

        # Resample to 1-minute intervals
        ohlc_df = trades_df['price'].resample('1min').ohlc()
        volume_df = trades_df['quantity'].resample('1min').sum()
        ohlc_df['volume'] = volume_df

        # Fill NaN values
        ohlc_df[['open', 'high', 'low', 'close']] = \
            ohlc_df[['open', 'high', 'low', 'close']].ffill()
        ohlc_df['volume'] = ohlc_df['volume'].fillna(0)

        # Validate OHLC invariants
        validation_passed = True

        # Check 1: Candlestick bars align with trade timestamps
        if not ohlc_df.empty:
            print(f"  ✓ Candlestick bars created: {len(ohlc_df)} 1-minute intervals")

            # Check 2: OHLC high >= max trade price in interval
            # Check 3: OHLC low <= min trade price in interval
            for idx, row in ohlc_df.iterrows():
                interval_start = idx
                interval_end = idx + timedelta(minutes=1)

                # Get trades in this interval
                interval_trades = trades_df.loc[
                    (trades_df.index >= interval_start) &
                    (trades_df.index < interval_end)
                ]

                if not interval_trades.empty:
                    max_trade_price = interval_trades['price'].max()
                    min_trade_price = interval_trades['price'].min()

                    assert row['high'] >= max_trade_price, \
                        f"OHLC high ({row['high']}) < max trade price ({max_trade_price})"
                    assert row['low'] <= min_trade_price, \
                        f"OHLC low ({row['low']}) > min trade price ({min_trade_price})"

            print("  ✓ OHLC high/low bounds validated")

            # Check 4: No candle with low > high
            assert (ohlc_df['low'] <= ohlc_df['high']).all(), \
                "Found candles with low > high"
            print("  ✓ All candles have low <= high")

            # Check 5: Spread never negative outside trade instants
            if 'l1_snapshots' in dataframes and not dataframes['l1_snapshots'].empty:
                l1_df = dataframes['l1_snapshots']
                # Check for negative spreads (should only happen during trades)
                negative_spreads = l1_df[l1_df['spread'] < 0]
                if len(negative_spreads) > 0:
                    print(f"  ⚠ Found {len(negative_spreads)} negative spreads (may occur during trades)")
                else:
                    print("  ✓ No negative spreads found")
        else:
            print("  ⚠ No trades to create OHLC data")
            validation_passed = False
    else:
        print("  ⚠ No trades executed in simulation")
        validation_passed = False

    print(f"\nOHLC Validation: {'PASSED' if validation_passed else 'FAILED'}")

    # 5. Generate dashboard and PDF report
    print("\nGenerating dashboard visualizations...")
    dashboard = DashboardGenerator()
    dashboard.generate_pdf_report(dataframes, engine.stats, OUTPUT_PDF)

    # 6. Final summary
    print("\n" + "=" * 70)
    print("SIMULATION COMPLETE - SUMMARY")
    print("=" * 70)
    print(f"Orders processed: {engine.stats['total_orders']:,}")
    print(f"Trades executed: {engine.stats['total_trades']:,}")
    print(f"Total volume: {engine.stats['total_volume']:,}")
    print(f"Maximum spread: {engine.stats['max_spread']:.4f}")
    print(f"Minimum spread: {engine.stats['min_spread']:.4f}")
    print(f"Validation errors: {engine.stats['validation_errors']}")
    print(f"Report generated: {OUTPUT_PDF}")

    if engine.stats['validation_errors'] == 0 and validation_passed:
        print("\n✅ All checks passed! Simulation pipeline is correct.")
    else:
        print(f"\n⚠ Found {engine.stats['validation_errors']} validation errors.")

    return engine, dataframes

# ============================================================================
# 7. EXECUTION BLOCK
# ============================================================================

if __name__ == "__main__":
    # Run the simulation
    engine, dataframes = run_simulation()

    # Optional: Display sample data
    print("\nSample Trade Data (first 5 trades):")
    if not dataframes['trades'].empty:
        print(dataframes['trades'].head())

    print("\nSample Order Book Snapshots (first 5):")
    if 'l1_snapshots' in dataframes and not dataframes['l1_snapshots'].empty:
        print(dataframes['l1_snapshots'].head())

ORDER BOOK SIMULATION - DAY 5 DELIVERABLE
Configuration:
  Seed: 42
  Orders to generate: 1000
  Output PDF: simulation_report.pdf

Starting simulation with 1000 orders...
  Processed 100/1000 orders...
  Processed 200/1000 orders...
  Processed 300/1000 orders...
  Processed 400/1000 orders...
  Processed 500/1000 orders...
  Processed 600/1000 orders...
  Processed 700/1000 orders...
  Processed 800/1000 orders...
  Processed 900/1000 orders...
  Processed 1000/1000 orders...
Simulation complete!
  Total orders: 1000
  Total trades: 817
  Total volume: 44849
  Validation errors: 0

Performing OHLC validation checks...
  ✓ Candlestick bars created: 100 1-minute intervals
  ✓ OHLC high/low bounds validated
  ✓ All candles have low <= high
  ✓ No negative spreads found

OHLC Validation: PASSED

Generating dashboard visualizations...
Generating PDF report: simulation_report.pdf
PDF report generated successfully: simulation_report.pdf

SIMULATION COMPLETE - SUMMARY
Orders processed: 1,000